In [1]:
from dataclasses import dataclass, field

from dowhy import CausalModel
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier

c:\Users\Roma\.virtualenvs\causal-promo-opt-N4J8iauW\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NUM_SIMULATIONS = 100

In [3]:
@dataclass
class ModelResult:
    treatment: any
    data: any
    outcome: str
    confounders: list
    method_params: dict

    model: any = field(init=False)
    estimand: any = field(init=False)
    estimate: any = field(init=False)

    def fit(self):
        self.model = CausalModel(data=self.data,
                                 treatment=self.treatment,
                                 outcome=self.outcome,
                                 common_causes=self.confounders)

        self.estimand = self.model.identify_effect()

        self.estimate = self.model.estimate_effect(
            identified_estimand=self.estimand,
            method_name="backdoor.econml.dml.LinearDML",
            method_params=self.method_params)

    def run_refuters(self):
        results = []

        refuters = [("data_subset", {
            "method_name": "data_subset_refuter",
            "subset_fraction": 0.8,
            "num_simulations": NUM_SIMULATIONS
        }),
                    ("random_common_cause", {
                        "method_name": "random_common_cause",
                        "num_simulations": NUM_SIMULATIONS
                    }),
                    ("random_treatment", {
                        "method_name": "placebo_treatment_refuter",
                        "num_simulations": NUM_SIMULATIONS
                    }),
                    ("random_outcome", {
                        "method_name": "dummy_outcome_refuter",
                        "num_simulations": NUM_SIMULATIONS
                    })]

        for name, params in refuters:
            refute = self.model.refute_estimate(self.estimand, self.estimate,
                                                **params)

            refute_res = refute[0] if isinstance(refute, list) else refute
            results.append({
                "treatment":
                self.treatment,
                "refuter":
                name,
                "original_effect":
                refute_res.estimated_effect,
                "new_effect":
                refute_res.new_effect,
                "delta":
                refute_res.new_effect - refute_res.estimated_effect,
                "p_value":
                refute_res.refutation_result.get("p_value", None)
            })

        return results

In [4]:
df = pd.read_csv('../data/software_usage_promotion.csv')

df['Both'] = (df['Discount'] *
                            df['Tech Support']).astype('category')
confounders = [
    'Size', 'Employee Count', 'PC Count', 'IT Spend', 'Major Flag',
    'Global Flag', 'Commercial Flag', 'SMC Flag'
]

In [5]:
tree_model = DecisionTreeClassifier(max_depth=3)

In [ ]:
model_obj = {}
model_results = {}

for i in ['Discount', 'Tech Support', 'Both']:
    model = ModelResult(treatment=i,
                        data=df,
                        outcome='Revenue',
                        confounders=confounders,
                        method_params={
                            'init_params': {
                                'model_y': GradientBoostingRegressor(),
                                'model_t': tree_model,
                                'discrete_treatment': True
                            },
                            'fit_params': {}
                        })
    model.fit()
    model_obj[i] = model
    res = model.run_refuters()
    model_results[i] = res

In [ ]:
model_results['Discount']

[{'treatment': 'Discount',
  'refuter': 'data_subset',
  'original_effect': np.float64(5017.959903077927),
  'new_effect': np.float64(4985.291268969297),
  'delta': np.float64(-32.66863410863061),
  'p_value': np.float64(0.9)},
 {'treatment': 'Discount',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(5017.959903077927),
  'new_effect': np.float64(4994.885767175298),
  'delta': np.float64(-23.074135902629678),
  'p_value': np.float64(0.74)},
 {'treatment': 'Discount',
  'refuter': 'random_treatment',
  'original_effect': np.float64(5017.959903077927),
  'new_effect': np.float64(-17.859696723002333),
  'delta': np.float64(-5035.81959980093),
  'p_value': np.float64(0.92)},
 {'treatment': 'Discount',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(-0.005110825578111573),
  'delta': np.float64(-0.005110825578111573),
  'p_value': np.float64(0.96)}]

In [ ]:
model_results['Tech Support']

[{'treatment': 'Tech Support',
  'refuter': 'data_subset',
  'original_effect': np.float64(6908.101437551587),
  'new_effect': np.float64(6704.559039357302),
  'delta': np.float64(-203.54239819428494),
  'p_value': np.float64(0.14)},
 {'treatment': 'Tech Support',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(6908.101437551587),
  'new_effect': np.float64(6731.574639273858),
  'delta': np.float64(-176.52679827772863),
  'p_value': np.float64(0.04)},
 {'treatment': 'Tech Support',
  'refuter': 'random_treatment',
  'original_effect': np.float64(6908.101437551587),
  'new_effect': np.float64(22.981628920806916),
  'delta': np.float64(-6885.11980863078),
  'p_value': np.float64(0.96)},
 {'treatment': 'Tech Support',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(-0.0008182651689924221),
  'delta': np.float64(-0.0008182651689924221),
  'p_value': np.float64(0.9)}]

In [ ]:
model_results['Both']

[{'treatment': 'Both',
  'refuter': 'data_subset',
  'original_effect': np.float64(8918.750346769628),
  'new_effect': np.float64(8693.991566623861),
  'delta': np.float64(-224.75878014576665),
  'p_value': np.float64(0.24)},
 {'treatment': 'Both',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(8918.750346769628),
  'new_effect': np.float64(8703.433089998358),
  'delta': np.float64(-215.31725677126997),
  'p_value': np.float64(0.08)},
 {'treatment': 'Both',
  'refuter': 'random_treatment',
  'original_effect': np.float64(8918.750346769628),
  'new_effect': np.float64(-4.7927222910895955),
  'delta': np.float64(-8923.543069060717),
  'p_value': np.float64(0.8999999999999999)},
 {'treatment': 'Both',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(-0.003566253922232223),
  'delta': np.float64(-0.003566253922232223),
  'p_value': np.float64(0.96)}]